# 案例 05 · HTTPS 证书过期，App 全量请求失败

> 沙箱实验：真实进程 + 阿里云 OpenAPI 模拟（响应带 `Mock: true`）。
> 云产品字段/命令与生产一致，资源规格与账单为模拟值。

## 场景

接入层证书到期未续签，客户端 TLS 握手报 `certificate has expired`，App 所有 HTTPS 请求失败；服务进程与端口均正常。

## 练习目标

- 用 `openssl s_client` 读取 `notAfter`
- 复现客户端证书校验失败
- 签发/替换证书并重启 HTTPS 服务
- 用 `cert_renew.py` 完成自动化巡检与续签

## 前置

- 仓库根目录执行 `python -m venv .venv && .venv/bin/pip install -r requirements.txt`
- 使用该 venv 的 Python kernel 打开本 Notebook；或运行 `tools/run_all_labs.py` 批量执行


In [ ]:
# 环境准备：定位仓库根目录并加载沙箱
import json, os, signal, ssl, socket, sys, time, gzip, shutil, urllib.parse, urllib.request
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / "labs" / "common" / "xhs_lab.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("请在 xhs-backend-cloud-ops-lab 仓库内运行本 Notebook")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from labs.common import Lab

lab = Lab("case-05-tls-cert")
lab.reset(start_api=True)
def cert_ok():
    try:
        ctx = ssl.create_default_context(cafile=str(lab.dir / "tls/server.crt"))
        with socket.create_connection(("127.0.0.1", lab.ports["api"]), timeout=3) as s:
            with ctx.wrap_socket(s, server_hostname="localhost"):
                return True
    except Exception:
        return False

lab.summary()


## 步骤 1 · 异常重现

下面注入一个可复现的真实故障，观察它对业务的影响。

In [ ]:
lab.inject_cert_expired()
time.sleep(1)
print("已注入：过期证书 + HTTPS 模式重启，cert_ok() =", cert_ok())


## 步骤 2 · 取证

按“入口 → 主机 → 进程 → 依赖 → 日志”的分层顺序收集证据，不要一上来就重启。

**证书有效期**

In [ ]:
print(lab.sh("openssl x509 -in %s -noout -subject -dates" % (lab.dir / "tls/server.crt")))

**证书订单**

In [ ]:
print(lab.aliyun("cas ListUserCertificateOrder"))

**客户端校验**

In [ ]:
print("证书校验通过" if cert_ok() else "证书校验失败：certificate has expired")

**服务日志**

In [ ]:
print(lab.sh("tail -n 3 /var/log/xhs/note-api.log"))

## 步骤 3 · 判断

把证据收敛成一句结论，决定止血动作。

In [ ]:
print("证书状态:", lab.state["certs"][0]["Status"], "notAfter:", lab.state["certs"][0]["NotAfter"])
print("客户端校验:", "通过" if cert_ok() else "失败")
print("判断：端口可连但 TLS 校验失败 → 证书问题，而非网络或进程问题")


## 步骤 4 · 手工处置（先止血）

执行最小必要的变更，优先恢复业务，再考虑根治。

In [ ]:
from labs.common.sims.certs import make_cert

# 1) 签发新证书（生产为 CAS 申请/上传）
make_cert(lab.dir / "tls/server.crt", lab.dir / "tls/server.key", days=90)
# 2) 重启 HTTPS 服务加载新证书（生产为 SLB 证书替换/热加载）
lab.restart_api(port=lab.ports["api"], tls=True)
lab.state["certs"][0].update({
    "NotAfter": time.strftime("%Y-%m-%d", time.localtime(time.time() + 86400 * 90)),
    "Status": "valid",
})
lab._save()
print("证书替换完成，客户端校验:", "通过" if cert_ok() else "失败")


## 步骤 5 · 验证

用可执行的断言确认恢复，而不是“看起来好了”。

In [ ]:
lab.check('HTTPS 校验通过', bool(cert_ok()))
lab.check('证书状态 valid', bool(lab.state['certs'][0]['Status'] == 'valid'))

## 步骤 6 · 自动化沉淀

脚本读取证书 `notAfter` 并做客户端校验，失败时签发新证书、重启 HTTPS 服务并再次校验证书链。

下面重置沙箱、重新注入故障，然后直接调用生产可用的自动化脚本（`--apply`；缺省为 dry-run）。

In [ ]:
lab.reset(start_api=True)
lab.inject_cert_expired()
time.sleep(1)
out = lab.auto('cert_renew.py', '--apply')
print(out)

In [ ]:
lab.check('自动化处置返回 REMEDIATED', 'RESULT: REMEDIATED' in out)
print('\n自动化脚本已可作为 cron / 事件总线触发器复用')

## 复盘与思考题

1. 证书过期前应设置几级提醒？分别提前多久？
2. 只更新了 SLB 证书但后端仍用旧证书，会有什么现象？
3. 如何避免证书私钥泄露？

## 收尾

沙箱资源会在本 Cell 中停止；重新运行请从第 1 个代码 Cell 开始。


In [ ]:
lab.stop_all()
print('沙箱已停止')